In [ ]:
# ======================
# IMPORT
# ======================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor

# ======================
# LOAD DATA
# ======================
df = pd.read_csv("train.csv")

target = "target"

# ======================
# EDA
# ======================
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum().sort_values(ascending=False).head())

# Distribution
sns.histplot(df[target], kde=True)
plt.title("Target Distribution")
plt.show()

# Correlation
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# ======================
# PREPARE DATA
# ======================
X = df.drop(columns=[target])
y = df[target]

# handle missing
X = X.fillna(X.mean())

# split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# scaling (สำคัญสำหรับ linear / ridge)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# ======================
# TRAIN MODELS
# ======================

# Linear Regression
model_lr = LinearRegression()
model_lr.fit(X_train_scaled, y_train)
pred_lr = model_lr.predict(X_valid_scaled)

# Ridge Regression (ดีกว่า linear ธรรมดา)
model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train_scaled, y_train)
pred_ridge = model_ridge.predict(X_valid_scaled)

# Gradient Boosting
model_gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3
)
model_gb.fit(X_train, y_train)
pred_gb = model_gb.predict(X_valid)

# ======================
# EVALUATE (MAE)
# ======================
mae_lr = mean_absolute_error(y_valid, pred_lr)
mae_ridge = mean_absolute_error(y_valid, pred_ridge)
mae_gb = mean_absolute_error(y_valid, pred_gb)

print("\n===== MAE Results =====")
print("Linear Regression:", mae_lr)
print("Ridge Regression:", mae_ridge)
print("Gradient Boosting:", mae_gb)

# ======================
# COMPARE
# ======================
results = pd.DataFrame({
    "Model": ["Linear", "Ridge", "GradientBoosting"],
    "MAE": [mae_lr, mae_ridge, mae_gb]
}).sort_values(by="MAE")

print("\n===== Sorted Results =====")
print(results)

# ======================
# BEST MODEL
# ======================
best_model_name = results.iloc[0]["Model"]

if best_model_name == "Linear":
    best_model = model_lr
elif best_model_name == "Ridge":
    best_model = model_ridge
else:
    best_model = model_gb

print("\nBest Model:", best_model_name)

# ======================
# PREDICT TEST
# ======================
test = pd.read_csv("test.csv")
test = test.fillna(test.mean())

if best_model_name in ["Linear", "Ridge"]:
    test_scaled = scaler.transform(test)
    pred_test = best_model.predict(test_scaled)
else:
    pred_test = best_model.predict(test)

# ======================
# SUBMISSION
# ======================
submission = pd.DataFrame({
    "id": test["id"],
    "target": pred_test
})

submission.to_csv("submission_new.csv", index=False)

print("\n✅ Saved: submission_new.csv")